In [ ]:
!pip install transformers scikit-learn librosa matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print("Imports done ✓")

In [ ]:
!wget "https://www.dropbox.com/s/sv94igp7zi3rsj1/mosi.pkl?dl=1" -O mosi.pkl
!ls -lh *.pkl

In [ ]:
import pickle
import numpy as np

with open('mosi.pkl', 'rb') as f:
    data = pickle.load(f)

print(type(data))
print(data.keys() if isinstance(data, dict) else dir(data))

In [ ]:
# Check if video features exist in MOSI pickle
sample = data['train'][0]
print("Number of elements in sample[0]:", len(sample[0]))
for i, x in enumerate(sample[0]):
    if hasattr(x, 'shape'):
        print(f"  [{i}] array shape: {x.shape}, dtype: {x.dtype}")
    else:
        print(f"  [{i}] type: {type(x).__name__}, value: {str(x)[:80]}")

In [ ]:
# Inspect structure of one split
sample = data['train'][0]
print(type(sample))
print(len(sample))
print()
for i, item in enumerate(sample):
    if hasattr(item, 'shape'):
        print(f"[{i}] array shape: {item.shape}")
    else:
        print(f"[{i}] {type(item).__name__}: {str(item)[:100]}")

In [ ]:
def extract_data(split):
    texts = []
    audio_features = []
    video_features = []
    labels = []

    for sample in data[split]:
        words = sample[0][0]
        audio = sample[0][1]
        video = sample[0][2]
        label = sample[1][0][0]

        texts.append(' '.join(words))
        audio_features.append(audio.mean(axis=0))
        video_features.append(video.mean(axis=0))
        labels.append(label)

    return texts, np.array(audio_features), np.array(video_features), np.array(labels)

train_texts, train_audio, train_video, train_labels = extract_data('train')
dev_texts, dev_audio, dev_video, dev_labels = extract_data('dev')
test_texts, test_audio, test_video, test_labels = extract_data('test')

train_y = (train_labels > 0).astype(int)
dev_y = (dev_labels > 0).astype(int)
test_y = (test_labels > 0).astype(int)

print(f"Audio features: {train_audio.shape}")
print(f"Video features: {train_video.shape}")
print("✓ All 3 modalities loaded")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Scale audio features
scaler = StandardScaler()
train_audio_scaled = scaler.fit_transform(train_audio)
dev_audio_scaled = scaler.transform(dev_audio)
test_audio_scaled = scaler.transform(test_audio)

# Train audio MLP
audio_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42)
audio_model.fit(train_audio_scaled, train_y)

audio_train_acc = accuracy_score(train_y, audio_model.predict(train_audio_scaled))
audio_test_acc = accuracy_score(test_y, audio_model.predict(test_audio_scaled))

print(f"Audio Baseline:")
print(f"  Train accuracy: {audio_train_acc:.3f}")
print(f"  Test accuracy:  {audio_test_acc:.3f}")

In [ ]:
from transformers import pipeline

text_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    top_k=3  # replaces return_all_scores in newer transformers
)

def get_text_scores(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = text_pipeline(batch, truncation=True, max_length=128)
        for r in results:
            # r is list of 3 dicts, sort by label to get [neg, neu, pos]
            scores = {item['label']: item['score'] for item in r}
            neg = scores.get('LABEL_0', 0)
            neu = scores.get('LABEL_1', 0)
            pos = scores.get('LABEL_2', 0)
            all_probs.append([neg, neu, pos])
        if i % 200 == 0:
            print(f"  Processed {i}/{len(texts)}")
    return np.array(all_probs)

print("Running text model on train...")
train_text_probs = get_text_scores(train_texts)
print("Running text model on test...")
test_text_probs = get_text_scores(test_texts)

train_text_preds = (train_text_probs[:, 2] > train_text_probs[:, 0]).astype(int)
test_text_preds = (test_text_probs[:, 2] > test_text_probs[:, 0]).astype(int)

print(f"\nText Baseline:")
print(f"  Train accuracy: {accuracy_score(train_y, train_text_preds):.3f}")
print(f"  Test accuracy:  {accuracy_score(test_y, test_text_preds):.3f}")

In [ ]:
# Debug: see exact output format
test_result = text_pipeline(["this is really great"], truncation=True, max_length=128)
print(test_result)
print(type(test_result[0]))

In [ ]:
# Re-run audio baseline with new extraction
audio_scaler = StandardScaler()
train_audio_scaled = audio_scaler.fit_transform(train_audio)
test_audio_scaled = audio_scaler.transform(test_audio)

audio_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42)
audio_model.fit(train_audio_scaled, train_y)

audio_test_acc = accuracy_score(test_y, audio_model.predict(test_audio_scaled))
print(f"Audio Baseline Test: {audio_test_acc:.3f}")

# Video baseline
video_scaler = StandardScaler()
train_video_scaled = video_scaler.fit_transform(train_video)
test_video_scaled = video_scaler.transform(test_video)

video_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42)
video_model.fit(train_video_scaled, train_y)

video_test_acc = accuracy_score(test_y, video_model.predict(test_video_scaled))
print(f"Video Baseline Test: {video_test_acc:.3f}")

print(f"\nResults so far:")
print(f"  Audio only:  {audio_test_acc:.3f}")
print(f"  Video only:  {video_test_acc:.3f}")
print(f"  Text only:   0.823")

In [ ]:
# Compute disagreement scores across all 3 modalities
audio_probs = audio_model.predict_proba(test_audio_scaled)  # [neg, pos]
video_probs = video_model.predict_proba(test_video_scaled)  # [neg, pos]
text_probs = test_text_probs[:, [0, 2]]                     # [neg, pos]

# Positive class probability from each modality
text_pos  = text_probs[:, 1]
audio_pos = audio_probs[:, 1]
video_pos = video_probs[:, 1]

# Disagreement = variance across the 3 modality scores
scores = np.stack([text_pos, audio_pos, video_pos], axis=1)
disagreement = np.var(scores, axis=1)

# Show top disagreement samples
top_idx = disagreement.argsort()[-15:][::-1]

print("TOP DISAGREEMENT SAMPLES (likely sarcasm/irony):\n")
for i in top_idx:
    true = 'POS' if test_y[i]==1 else 'NEG'
    print(f"'{test_texts[i]}'")
    print(f"  Label:{true} | Text:{text_pos[i]:.2f} | Audio:{audio_pos[i]:.2f} | Video:{video_pos[i]:.2f} | Disagreement:{disagreement[i]:.3f}")
    print()

In [ ]:
# Fusion model WITH disagreement score
from sklearn.neural_network import MLPClassifier

# Stack all modality features + disagreement
X_train_fusion = np.hstack([
    train_audio_scaled,
    train_video_scaled,
    train_text_probs[:, [0, 2]],  # neg, pos text probs
])

X_test_fusion = np.hstack([
    test_audio_scaled,
    test_video_scaled,
    test_text_probs[:, [0, 2]],
])

# Train disagreement scores on train set too
train_text_pos  = train_text_probs[:, 2]
train_audio_pos = audio_model.predict_proba(train_audio_scaled)[:, 1]
train_video_pos = video_model.predict_proba(train_video_scaled)[:, 1]
train_scores    = np.stack([train_text_pos, train_audio_pos, train_video_pos], axis=1)
train_disagreement = np.var(train_scores, axis=1).reshape(-1, 1)
test_disagreement  = disagreement.reshape(-1, 1)

# Fusion WITHOUT disagreement
fusion_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
fusion_model.fit(X_train_fusion, train_y)
fusion_acc = accuracy_score(test_y, fusion_model.predict(X_test_fusion))

# Fusion WITH disagreement
X_train_fusion_d = np.hstack([X_train_fusion, train_disagreement])
X_test_fusion_d  = np.hstack([X_test_fusion, test_disagreement])

fusion_d_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
fusion_d_model.fit(X_train_fusion_d, train_y)
fusion_d_acc = accuracy_score(test_y, fusion_d_model.predict(X_test_fusion_d))

print("=" * 45)
print("FINAL RESULTS TABLE")
print("=" * 45)
print(f"Audio only:                    {audio_test_acc:.3f}")
print(f"Video only:                    {video_test_acc:.3f}")
print(f"Text only:                     0.823")
print(f"Fusion (no disagreement):      {fusion_acc:.3f}")
print(f"Fusion + disagreement score:   {fusion_d_acc:.3f}")
print("=" * 45)

In [ ]:
# The real finding: error analysis on high vs low disagreement samples
median_disagreement = np.median(disagreement)

high_d_idx = disagreement >= median_disagreement
low_d_idx  = disagreement < median_disagreement

# Text-only accuracy on high vs low disagreement subsets
text_preds = test_text_preds

text_acc_low  = accuracy_score(test_y[low_d_idx],  text_preds[low_d_idx])
text_acc_high = accuracy_score(test_y[high_d_idx], text_preds[high_d_idx])

fusion_acc_low  = accuracy_score(test_y[low_d_idx],  fusion_model.predict(X_test_fusion[low_d_idx]))
fusion_acc_high = accuracy_score(test_y[high_d_idx], fusion_model.predict(X_test_fusion[high_d_idx]))

print("ACCURACY BY DISAGREEMENT LEVEL")
print(f"{'':30} {'Low Disagree':>15} {'High Disagree':>15}")
print(f"{'Text only':30} {text_acc_low:>15.3f} {text_acc_high:>15.3f}")
print(f"{'Fusion':30} {fusion_acc_low:>15.3f} {fusion_acc_high:>15.3f}")
print(f"\nSamples:  Low={low_d_idx.sum()}  High={high_d_idx.sum()}")
print(f"\nKey insight: when modalities disagree strongly,")
print(f"text accuracy drops {text_acc_low - text_acc_high:.3f} points")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("When Modalities Disagree: CMU-MOSI Results\nSean Bell — CS466", fontsize=13)

# Plot 1: Unimodal accuracy bar chart
models = ['Audio\nonly', 'Video\nonly', 'Text\nonly', 'Fusion', 'Fusion+\nDisagreement']
accs   = [0.502, 0.485, 0.823, 0.790, 0.762]
colors = ['#e74c3c','#e74c3c','#2ecc71','#3498db','#9b59b6']
axes[0].bar(models, accs, color=colors)
axes[0].set_ylim(0.4, 0.9)
axes[0].set_title('Model Accuracy by Modality')
axes[0].set_ylabel('Test Accuracy')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')

# Plot 2: Accuracy by disagreement level
x = np.arange(2)
width = 0.35
axes[1].bar(x - width/2, [0.789, 0.857], width, label='Text only', color='#2ecc71')
axes[1].bar(x + width/2, [0.784, 0.796], width, label='Fusion',     color='#3498db')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Low Disagreement', 'High Disagreement'])
axes[1].set_ylim(0.7, 0.9)
axes[1].set_title('Accuracy by Disagreement Level')
axes[1].set_ylabel('Test Accuracy')
axes[1].legend()

# Plot 3: Disagreement score distribution
axes[2].hist(disagreement, bins=30, color='#9b59b6', edgecolor='white')
axes[2].set_title('Disagreement Score Distribution')
axes[2].set_xlabel('Variance across modality predictions')
axes[2].set_ylabel('Count')
axes[2].axvline(np.median(disagreement), color='red', linestyle='--', label='Median')
axes[2].legend()

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved results.png")

from google.colab import files
files.download('results.png')

In [ ]:
# ============================================================
# DISAGREEMENT-AWARE 3-CLASS MODEL
# Class 0 = Negative, Class 1 = Positive, Class 2 = Ambiguous
# ============================================================

# Compute train disagreement scores
train_text_pos  = train_text_probs[:, 2]
train_audio_pos = audio_model.predict_proba(train_audio_scaled)[:, 1]
train_video_pos = video_model.predict_proba(train_video_scaled)[:, 1]
train_scores    = np.stack([train_text_pos, train_audio_pos, train_video_pos], axis=1)
train_disagreement = np.var(train_scores, axis=1)

# Threshold: top 25% disagreement = ambiguous
threshold = np.percentile(train_disagreement, 75)
print(f"Disagreement threshold (75th percentile): {threshold:.4f}")

# Create 3-class labels
train_y_3class = np.where(train_disagreement > threshold, 2, train_y)
test_y_3class  = np.where(disagreement > threshold, 2, test_y)

print(f"Train class distribution:")
print(f"  Negative (0):  {(train_y_3class==0).sum()}")
print(f"  Positive (1):  {(train_y_3class==1).sum()}")
print(f"  Ambiguous (2): {(train_y_3class==2).sum()}")

# Build feature matrix with disagreement score
X_train_3 = np.hstack([
    train_text_probs[:, [0,2]],
    train_audio_scaled,
    train_video_scaled,
    train_disagreement.reshape(-1,1)
])

X_test_3 = np.hstack([
    test_text_probs[:, [0,2]],
    test_audio_scaled,
    test_video_scaled,
    disagreement.reshape(-1,1)
])

# Train 3-class model
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

model_3class = MLPClassifier(hidden_layer_sizes=(128,64),
                              max_iter=300, random_state=42)
model_3class.fit(X_train_3, train_y_3class)
preds_3class = model_3class.predict(X_test_3)

print("\n3-CLASS MODEL RESULTS:")
print(classification_report(
    test_y_3class,
    preds_3class,
    target_names=['Negative', 'Positive', 'Ambiguous']
))

# Key comparison: on ambiguous samples only,
# how does 3-class model compare to forcing binary?
ambiguous_mask = test_y_3class == 2

# Binary model forced prediction on ambiguous samples
binary_on_ambiguous = accuracy_score(
    test_y[ambiguous_mask],
    fusion_model.predict(X_test_fusion[ambiguous_mask])
)

# 3-class model: correct if it says "ambiguous"
ambiguous_correct = (preds_3class[ambiguous_mask] == 2).mean()

print(f"\nOn high-disagreement samples ({ambiguous_mask.sum()} samples):")
print(f"  Binary fusion accuracy:           {binary_on_ambiguous:.3f}")
print(f"  3-class model flags as ambiguous: {ambiguous_correct:.3f}")
print(f"\nKey insight: the 3-class model correctly identifies")
print(f"ambiguous samples {ambiguous_correct*100:.1f}% of the time,")
print(f"vs binary model which is forced to guess and gets {binary_on_ambiguous*100:.1f}%")

In [ ]:
# Show what ambiguous samples actually look like
correctly_flagged = (preds_3class == 2) & (test_y_3class == 2)
indices = np.where(correctly_flagged)[0]

print(f"Samples correctly identified as AMBIGUOUS: {correctly_flagged.sum()}\n")
print("Examples:\n")
for i in indices[:10]:
    print(f"'{test_texts[i]}'")
    print(f"  Text:{text_pos[i]:.2f} | Audio:{audio_pos[i]:.2f} | Video:{video_pos[i]:.2f} | δ:{disagreement[i]:.3f}")
    print()

In [ ]:
i = np.argmax(disagreement)
print(f"'{test_texts[i]}'")
print(f"Text: {text_pos[i]:.2f} | Audio: {audio_pos[i]:.2f} | Video: {video_pos[i]:.2f}")
print(f"Disagreement δ: {disagreement[i]:.3f}")

In [ ]:
idx = np.where(correctly_flagged)[0][0]
print(f"'{test_texts[idx]}'")
print(f"3-class model prediction: AMBIGUOUS")
print(f"δ = {disagreement[idx]:.3f}")